In [0]:
from pyspark.sql.functions import lead, col
from pyspark.sql.window import Window

class Transform:
    def __init__(self):
        pass
    
    def transform(self,inputDFs):
        pass


            

In [0]:
class FirstTranformer(Transform):
    def transform(self,inputDFs):
        # customer buying Airpods after buying Iphone

        #print("----- FirstTranformer ----")
        transactionInputDF = inputDFs.get("transactionInputdf")
        #transactionInputDF.show()

        #Get current and next product for each customer
        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")
        tranformedDF = transactionInputDF.withColumn("next_product_name",lead("product_name").over(windowSpec))        
        
        # Retrieve only customers who have bought iPhone and later Airpods as next product
        filteredDF = tranformedDF.orderBy("customer_id","transaction_date","product_name").filter(
            (col("product_name") == 'iPhone') & (col("next_product_name") == 'AirPods')
            )
        
        #filteredDF.orderBy("customer_id","transaction_date","product_name").show()

        customerInputdf = inputDFs.get("customerInputdf")

        joineddf = filteredDF.join(
            customerInputdf,
            "customer_id",
            "inner"
        )
        
        return joineddf.select("customer_id","customer_name","location").orderBy("customer_id")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, broadcast, collect_set, size, array_contains

class SecondTranformer(Transform):
    def transform(self,inputDFs):
        # customer  Airpods and Iphone 

        #print("----- FirstTranformer ----")
        transactionInputDF = inputDFs.get("transactionInputdf")
        #transactionInputDF.show()

        # Retrieve  customers who have bought iPhone and Airpods
        groupdf = transactionInputDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        print("Grouped DF")
        groupdf.show()

        filteredDF = groupdf.filter(
            (array_contains(col("products"), "iPhone")) &
            (array_contains(col("products"), "AirPods")) & 
            (size(col("products")) == 2)
        )
        
        print("Only Airpods and iPhone")
        filteredDF.show()

        customerInputdf = inputDFs.get("customerInputdf")

        customerInputdf.show()

        joinDF =  customerInputdf.join(
           broadcast(filteredDF),
            "customer_id"
        )

        print("JOINED DF")
        joinDF.show()

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, broadcast, collect_set, size, array_contains, row_number

class AllProductsExceptInitialTranformer(Transform):
    def transform(self,inputDFs):

        # get all transactions from transactions from transactionInputdf
        transactionInputDF = inputDFs.get("transactionInputdf")
        #transactionInputDF.show()

        # retrieve all products bought by customers and order them by transaction_date and further get all products bought by each customer except first(initial) product
        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")
        tranformedDF = transactionInputDF.withColumn("row_num", row_number().over(windowSpec))

        #tranformedDF.display()  
   
        # retrieve all transactions except first transaction
        filteredDF = tranformedDF.filter(col("row_num") > 1)

        #filteredDF.display()

        # remove duplicate product names bought by each customer
        groupdf = filteredDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        #groupdf.display()

        # get customer details
        customerInputdf = inputDFs.get("customerInputdf")

        # retrieve all customer and products bought by each customer
        joineddf = groupdf.join(
            customerInputdf,
            "customer_id",
            "inner"
        ).select("customer_id","customer_name","location","products")

        #joineddf.display()

        return joineddf

 

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, min, datediff, abs, avg

class TimeBetweenIphoneAndAirPodsTranformer(Transform):
    def transform(self,inputDFs):
        transactionInputDF = inputDFs.get("transactionInputdf")

        # 1. Filter for relevant products
        filteredDF = transactionInputDF.filter(col("product_name").isin(["iPhone", "AirPods"]))

        # 2. Get the first purchase date for each customer and product
        windowSpec = Window.partitionBy("customer_id", "product_name").orderBy("transaction_date")

        first_purchases = (
            filteredDF
            .withColumn("row_num", row_number().over(windowSpec))
            .filter(col("row_num") == 1)
        )

        # 3. Pivot to get 'iPhone' and 'AirPods' dates into separate columns
        pivotedDF = (
            first_purchases
            .groupBy("customer_id")
            .pivot("product_name", ["iPhone", "AirPods"])
            .agg(min("transaction_date"))
        )

        # 4. Filter for customers who bought BOTH, then calculate lag in days
        lagDF = (
            pivotedDF
            .filter(col("iPhone").isNotNull() & col("AirPods").isNotNull())
            .withColumn("lag_days", abs(datediff(col("AirPods"), col("iPhone"))))
        )

        # 5. Calculate the average lag time across all customers
        avg_lag = lagDF.select(avg("lag_days").alias("avg_lag_days"))

        #avg_lag.display()

        return avg_lag


In [0]:
#Identify the top 3 selling products in each category by total revenue
from pyspark.sql.functions import sum, dense_rank
from pyspark.sql.window import Window

class Top3PerCategoryTranformer(Transform):
    def transform(self,inputDFs):
        transactionInputDF = inputDFs.get("transactionInputdf")
        productInputdf= inputDFs.get("productInputdf")

        transactionInputDF.display()
        productInputdf.display()

        print("-------------------------------------------")

        # 1. Join transactions with product to get category and price
        salesDF = (
            transactionInputDF
            .join(productInputdf, "product_name")
        )

        salesDF.display()

        # 2. Calculate total revenue per category and product
        revenueDF = (
            salesDF
            .groupBy("category")
            .agg(sum("price").alias("total_revenue"))
        )

        revenueDF.display()

        # 3. Rank products within each category by revenue
        windowSpec = Window.partitionBy("category").orderBy(col("total_revenue").desc())

        top3_per_category = (
            revenueDF
            .withColumn("rank", dense_rank().over(windowSpec))
            .filter(col("rank") <= 3)
            .orderBy("category", "rank")
        )

        top3_per_category.display()
        return top3_per_category